# MERCon one-by-one evaluation notebook

Run this on Kaggle after attaching your project/dataset bundle and checkpoint. This notebook evaluates datasets one by one, so you can stop or skip cells without losing previous outputs.

Do not rerun old baseline tables here. This notebook is for missing/new camera-ready evidence: ARR LPIPS, extra paired datasets, and optional no-reference qualitative outputs.

In [ ]:
# Cell 1: setup, source/checkpoint/dataset discovery
from pathlib import Path
import os, shutil, subprocess, sys, json

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
OUT = WORK / 'mercon_one_by_one_outputs'
OUT.mkdir(exist_ok=True)

def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

eval_files = sorted(INPUT.rglob('evaluation.py'))
if eval_files:
    SRC = eval_files[0].parent
    print('Source detected from Kaggle input:', SRC)
else:
    # Fallback: use the public GitHub repo for project code.
    # Kaggle Internet must be ON for this path. Datasets still need to be attached separately.
    REPO = WORK / 'Diffusion_new_final'
    if not REPO.exists():
        subprocess.run(['git', 'clone', 'https://github.com/chirana07/Diffusion_new_final.git', str(REPO)], check=True)
    SRC = REPO
    print('Source cloned from GitHub:', SRC)
    # If git-lfs exists, try to materialize final.pth from the repo. This may fail if LFS/network is unavailable.
    try:
        subprocess.run(['git', 'lfs', 'install'], cwd=SRC, check=False)
        subprocess.run(['git', 'lfs', 'pull'], cwd=SRC, check=False)
    except Exception as exc:
        print('git-lfs pull skipped/failed:', exc)

# Copy only source files to writable working dir; do not copy huge datasets.
CODE = WORK / 'src_mercon_eval'
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir()
for p in SRC.glob('*.py'):
    shutil.copy2(p, CODE / p.name)
print('Code copied to:', CODE)

# Locate evaluation data. Prefer canonical Datasets/, but also support the manual Zip_file/ layout.
# If Kaggle exposes the upload only as a .zip file, unzip it to /kaggle/working first.
def find_eval_data(search_roots):
    dataset_dirs = []
    zip_file_dirs = []
    for root in search_roots:
        dataset_dirs.extend([p for p in root.rglob('Datasets') if p.is_dir()])
        zip_file_dirs.extend([p for p in root.rglob('Zip_file') if p.is_dir()])
    if dataset_dirs:
        return dataset_dirs[0], 'canonical_datasets'
    if zip_file_dirs:
        return zip_file_dirs[0], 'manual_zip_file'
    return None, None

DATA, DATA_LAYOUT = find_eval_data([INPUT])
if DATA is None:
    zips = sorted(INPUT.rglob('*.zip'))
    if zips:
        UNZIPPED = WORK / 'unzipped_kaggle_inputs'
        UNZIPPED.mkdir(exist_ok=True)
        for z in zips:
            print('Unzipping input archive:', z)
            subprocess.run(['unzip', '-q', '-o', str(z), '-d', str(UNZIPPED)], check=True)
        DATA, DATA_LAYOUT = find_eval_data([UNZIPPED])

if DATA is None:
    raise FileNotFoundError('Could not find Datasets/ or Zip_file/ under /kaggle/input, even after unzipping input archives.')
print('Evaluation data detected:', DATA)
print('Data layout:', DATA_LAYOUT)

# Locate checkpoint. Prefer distilled/best names; fallback to final.pth.
# If a checkpoint is uploaded as a .zip, unzip it first.
CKPT_UNZIP = WORK / 'unzipped_checkpoints'
for z in sorted(INPUT.rglob('*.zip')):
    if any(k in z.name.lower() for k in ('distill', 'ckpt', 'checkpoint')):
        CKPT_UNZIP.mkdir(exist_ok=True)
        print('Unzipping checkpoint archive:', z)
        subprocess.run(['unzip', '-q', '-o', str(z), '-d', str(CKPT_UNZIP)], check=True)

ckpts = []
for search_root in [INPUT, CKPT_UNZIP, SRC]:
    if not search_root.exists():
        continue
    for p in search_root.rglob('*'):
        if p.is_file() and p.suffix.lower() in {'.pth', '.pt', '.ckpt'}:
            name = p.name.lower()
            score = 0
            path_text = str(p).lower()
            if str(p).startswith(str(INPUT)) or str(p).startswith(str(CKPT_UNZIP)):
                score += 50  # prefer explicitly attached Kaggle checkpoint datasets over repo pointers
            if 'distill' in name or 'distill' in path_text: score += 100
            if 'best' in name: score += 20
            if 'k5' in name: score += 10
            if name == 'final.pth': score += 5
            ckpts.append((score, p))
if not ckpts:
    raise FileNotFoundError('No .pth/.pt/.ckpt found under /kaggle/input. Attach best_distill_K5.pth or lumidiff-distill.zip.')
print('Checkpoint candidates:')
for score, pth in sorted(ckpts, reverse=True):
    print(f'  score={score:3d} size_MB={pth.stat().st_size/1024/1024:.1f} path={pth}')
CHECKPOINT = sorted(ckpts, reverse=True)[0][1]
print('Checkpoint selected:', CHECKPOINT)
if CHECKPOINT.stat().st_size < 1024 * 1024:
    print('WARNING: selected checkpoint is very small. It may be a Git LFS pointer, not real weights.')
    print('If this happens, attach best_distill_K5.pth as a Kaggle dataset and set CHECKPOINT manually.')
if 'distill' not in str(CHECKPOINT).lower():
    print('WARNING: checkpoint path does not contain distill. Verify this is the FSD student checkpoint.')

def p(*parts):
    return str(DATA.joinpath(*parts))

if DATA_LAYOUT == 'manual_zip_file':
    SPLITS = {
        'lol_v1_eval15': (p('LOL-v1 eval15 ', 'low'), p('LOL-v1 eval15 ', 'high')),
        'lolv2_real': (p('LOL-v2 Real Test                   ', 'Low'), p('LOL-v2 Real Test                   ', 'Normal')),
        'lolv2_syn': (p('LOL-v2 Synthetic Test', 'Low'), p('LOL-v2 Synthetic Test', 'Normal')),
        'lsrw_huawei': (p('Huawei', 'low'), p('Huawei', 'high')),
        'lsrw_nikon': (p('Nikon', 'low'), p('Nikon', 'high')),
        # Your current Zip_file.zip does not contain VE-LOL synthetic Normal_test. Leave this visible but unavailable.
        've_lol_syn': (p('VE-LOL-L-Syn-Low_test'), p('VE-LOL-L-Syn-Normal_test')),
    }
else:
    SPLITS = {
        'lol_v1_eval15': (p('lol_dataset','eval15','low'), p('lol_dataset','eval15','high')),
        'lolv2_real': (p('LOL-v2','Real_captured','Test','Low'), p('LOL-v2','Real_captured','Test','Normal')),
        'lolv2_syn': (p('LOL-v2','Synthetic','Test','Low'), p('LOL-v2','Synthetic','Test','Normal')),
        'lsrw_huawei': (p('LSRW','Huawei','low'), p('LSRW','Huawei','high')),
        'lsrw_nikon': (p('LSRW','Nikon','low'), p('LSRW','Nikon','high')),
        'comp': (p('COMP','Denoised_LLIE_val_in'), p('COMP','Denoised_LLIE_val_gt')),
        've_lol_syn': (p('VE-LOL-L','VE-LOL-L-Syn','VE-LOL-L-Syn-Low_test'), p('VE-LOL-L','VE-LOL-L-Syn','VE-LOL-L-Syn-Normal_test')),
        # VE-LOL captured appears byte-identical to LOL-v2 Real in the local audit, so do not use as independent evidence.
        've_lol_cap_duplicate_check_only': (p('VE-LOL-L','VE-LOL-L-Cap-Full','VE-LOL-L-Cap-Low_test'), p('VE-LOL-L','VE-LOL-L-Cap-Full','VE-LOL-L-Cap-Normal_test')),
    }

for name, (low, high) in SPLITS.items():
    print(f'{name:30s}', Path(low).exists(), Path(high).exists(), low, '|', high)

print('Output root:', OUT)

In [ ]:
# Cell 2: install dependencies and define evaluation helper
!pip install -q lpips fvcore

import pandas as pd

# Patch evaluator to skip non-aligned low/high image pairs, e.g. a few LSRW files.
# Resizing GT would make PSNR/SSIM less defensible, so skipped pairs are reported via n.
eval_path = CODE / 'evaluation.py'
eval_text = eval_path.read_text()
old_block = """        low_pil = Image.open(low_path).convert(\"RGB\")
        high_pil = Image.open(high_path).convert(\"RGB\")

        low_padded, (orig_w, orig_h) = pad_to_multiple_of_8(low_pil)
"""
new_block = """        low_pil = Image.open(low_path).convert(\"RGB\")
        high_pil = Image.open(high_path).convert(\"RGB\")
        if low_pil.size != high_pil.size:
            print(
                f"[{split_name}] skipping {name}: low size {low_pil.size} "
                f"!= high size {high_pil.size}"
            )
            continue

        low_padded, (orig_w, orig_h) = pad_to_multiple_of_8(low_pil)
"""
if old_block in eval_text:
    eval_path.write_text(eval_text.replace(old_block, new_block))
    print('Patched evaluation.py to skip mismatched low/high dimensions.')
else:
    print('Dimension-skip patch already applied or source changed.')

def run_eval_one(split_name, steps=5, alpha=0.5, lpips=True):
    low, high = SPLITS[split_name]
    if not Path(low).is_dir() or not Path(high).is_dir():
        raise FileNotFoundError(f'Missing low/high folder for {split_name}: {low} | {high}')
    tag = f'{split_name}_s{steps}_a{int(alpha*100):03d}'
    cmd = [
        sys.executable, 'evaluation.py',
        '--splits', f'{split_name}:{low}:{high}',
        '--inference-steps', str(steps),
        '--sampler', 'ddim',
        '--checkpoint', str(CHECKPOINT),
        '--gate-alpha', str(alpha),
        '--gate-floor', '0.5',
        '--results-root', str(OUT / 'eval'),
        '--tag', tag,
    ]
    if not lpips:
        cmd.append('--no-lpips')
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=CODE, check=True)
    summary = OUT / f'eval_{tag}' / 'summary.csv'
    print('Summary:', summary)
    display(pd.read_csv(summary))
    return summary

def show_existing_outputs():
    for s in sorted(OUT.glob('eval_*/summary.csv')):
        print('\n', s)
        display(pd.read_csv(s))


## Cell 3: LOL-v2 Real ARR LPIPS

Run this even if you already have LOL-v2 Real PSNR/SSIM, because the reviewer specifically complained that ARR LPIPS was not directly measured for `alpha=0.5`.

In [ ]:
run_eval_one('lolv2_real', steps=5, alpha=0.5, lpips=True)

## Cell 4: LOL-v2 Synthetic

Reviewer requested more datasets. This is paired and directly usable.

In [ ]:
run_eval_one('lolv2_syn', steps=5, alpha=0.5, lpips=True)

## Cell 5: LOL-v1 eval15

This gives revised/raw-folder eval15 evidence. It is small and should run quickly.

In [ ]:
run_eval_one('lol_v1_eval15', steps=5, alpha=0.5, lpips=True)

## Cell 6: LSRW Huawei

Paired 30-image subset. Use as additional cross-dataset evidence.

In [ ]:
run_eval_one('lsrw_huawei', steps=5, alpha=0.5, lpips=True)

## Cell 7: LSRW Nikon

Paired 20-image subset. Use as additional cross-dataset evidence.

In [ ]:
run_eval_one('lsrw_nikon', steps=5, alpha=0.5, lpips=True)

## Cell 8: COMP validation set

This folder is paired (`Denoised_LLIE_val_in` and `Denoised_LLIE_val_gt`). Run it as extra paired evidence, but in the paper describe it only if you can identify the dataset/protocol clearly.

In [ ]:
run_eval_one('comp', steps=5, alpha=0.5, lpips=True)

## Cell 9: VE-LOL Synthetic

This is paired. Run it only if you want a VE-LOL row. Do not count VE-LOL Captured as independent if it duplicates LOL-v2 Real.

In [ ]:
run_eval_one('ve_lol_syn', steps=5, alpha=0.5, lpips=True)

## Cell 10: no-reference qualitative inference helper

Use this only for folders without ground truth. These outputs are for qualitative figures or supplementary evidence. Do not report PSNR/SSIM/LPIPS for these folders.

In [ ]:
import sys, glob, random
sys.path.insert(0, str(CODE))
import torch
import torchvision.transforms.functional as TF
from PIL import Image
from tqdm import tqdm
from config import Config
from model import ResidualConditionedUNet
from diffusion import DiffusionEngine

_NR_MODEL = None
_NR_DIFF = None
_NR_DEVICE = None

def _load_no_ref_model():
    global _NR_MODEL, _NR_DIFF, _NR_DEVICE
    if _NR_MODEL is not None:
        return _NR_MODEL, _NR_DIFF, _NR_DEVICE
    conf = Config()
    _NR_DEVICE = conf.DEVICE
    ckpt = torch.load(CHECKPOINT, map_location=_NR_DEVICE)
    use_prior = bool(ckpt.get('use_illum_prior', False)) if isinstance(ckpt, dict) else False
    model = ResidualConditionedUNet(use_illum_prior=use_prior).to(_NR_DEVICE)
    if isinstance(ckpt, dict) and 'ema' in ckpt:
        model.load_state_dict(ckpt['ema'])
    elif isinstance(ckpt, dict) and 'model' in ckpt:
        model.load_state_dict(ckpt['model'])
    else:
        model.load_state_dict(ckpt)
    model.eval()
    _NR_MODEL = model
    _NR_DIFF = DiffusionEngine()
    return _NR_MODEL, _NR_DIFF, _NR_DEVICE

def _pad8(img):
    w, h = img.size
    nw = ((w + 7) // 8) * 8
    nh = ((h + 7) // 8) * 8
    return TF.pad(img, [0, 0, nw - w, nh - h], fill=0), (w, h)

def run_no_ref_folder(name, folder, limit=None, steps=5, alpha=0.5):
    folder = Path(folder)
    if not folder.is_dir():
        raise FileNotFoundError(folder)
    image_paths = sorted([p for p in folder.rglob('*') if p.suffix.lower() in {'.png','.jpg','.jpeg','.bmp'}])
    if limit is not None:
        image_paths = image_paths[:limit]
    out_dir = OUT / 'qualitative_no_reference' / name
    out_dir.mkdir(parents=True, exist_ok=True)
    model, diff, device = _load_no_ref_model()
    print(f'{name}: {len(image_paths)} images -> {out_dir}')
    for img_path in tqdm(image_paths):
        img = Image.open(img_path).convert('RGB')
        padded, (ow, oh) = _pad8(img)
        low = (TF.to_tensor(padded) - 0.5) * 2.0
        low = low.unsqueeze(0).to(device)
        with torch.no_grad():
            pred = diff.ddim_sample(model, low, inference_steps=steps, gate_alpha=alpha, gate_floor=0.5)
        pred = torch.clamp((pred + 1.0) / 2.0, 0.0, 1.0)
        out = TF.to_pil_image(pred.squeeze(0).cpu()).crop((0, 0, ow, oh))
        rel = img_path.relative_to(folder)
        save_path = out_dir / rel
        save_path.parent.mkdir(parents=True, exist_ok=True)
        save_path = save_path.with_name('enhanced_' + save_path.name)
        out.save(save_path)
    print('Saved qualitative outputs:', out_dir)
    return out_dir

## Cell 11: DICM, LIME, MEF, NPE qualitative outputs

These folders have no paired ground truth in the current dataset layout. Run this cell only if you want examples for a qualitative/generalization figure.

In [ ]:
run_no_ref_folder('dicm', DATA / 'DICM', limit=None, steps=5, alpha=0.5)
run_no_ref_folder('lime', DATA / 'LIME', limit=None, steps=5, alpha=0.5)
run_no_ref_folder('mef', DATA / 'MEF', limit=None, steps=5, alpha=0.5)
run_no_ref_folder('npe', DATA / 'NPE', limit=None, steps=5, alpha=0.5)

## Cell 12: ExDark qualitative sample

ExDark is large and has no restoration ground truth here. Run a sample first. Increase `limit` only if you need more qualitative candidates.

In [ ]:
# First pass: 10 images per class folder = manageable qualitative candidates.
for class_dir in sorted((DATA / 'ExDark').iterdir()):
    if class_dir.is_dir():
        run_no_ref_folder('exdark_' + class_dir.name.lower(), class_dir, limit=10, steps=5, alpha=0.5)

## Cell 13: collect all summaries and outputs

Run this after the evaluation cells you choose. Download `mercon_one_by_one_outputs.zip` from `/kaggle/working`.

In [ ]:
show_existing_outputs()
!cd /kaggle/working && zip -r mercon_one_by_one_outputs.zip mercon_one_by_one_outputs >/dev/null
print('Download: /kaggle/working/mercon_one_by_one_outputs.zip')